# From the perceptron to logistic regression
### Lecture demo, CSC 371. 

In [ ]:
%matplotlib inline
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import (LinearRegression, LogisticRegression,
                                  Perceptron, SGDClassifier, SGDRegressor)

warnings.filterwarnings('ignore')          # keeps convergence chatter off the screen

df = pd.read_csv('iris.data', header=None, encoding='utf-8')

# Versicolor vs virginica, petal length and petal width. These two overlap.
pair = df.iloc[50:150]
X = pair.iloc[:, [2, 3]].values.astype(float)
y = np.where(pair.iloc[:, 4].values == 'Iris-versicolor', 0, 1)

# The file lists all 50 versicolor and then all 50 virginica. Shuffle once, so
# the perceptron is not fed one entire class at a time.
order = np.random.RandomState(1).permutation(len(X))
X, y = X[order], y[order]

X_std = (X - X.mean(axis=0)) / X.std(axis=0)
y_pm = np.where(y == 1, 1.0, -1.0)         # the same labels as -1 / +1

print(f'{len(X)} flowers, {np.bincount(y)} per class')

In [ ]:
def show_regions(model, features, labels, ax, title):
    """Shade the plane by predicted class and scatter the data on top."""
    DecisionBoundaryDisplay.from_estimator(
        model, features, ax=ax, alpha=0.3, cmap='coolwarm',
        response_method='predict')
    for value, colour, marker in [(labels.min(), 'tab:blue', 'o'),
                                  (labels.max(), 'tab:red', 's')]:
        spot = labels == value
        ax.scatter(features[spot, 0], features[spot, 1], c=colour, marker=marker,
                   s=28, edgecolor='black', linewidth=0.4, label=f'class {value:g}')
    ax.set_xlabel('Petal length')
    ax.set_ylabel('Petal width')
    ax.set_title(title)

## 1. Perceptron

In [ ]:
EARLY, LATE, EPOCHS = 25, 55, 60


def perceptron_after(epochs):
    """A perceptron trained for exactly this many passes over the data."""
    model = Perceptron(random_state=0)
    for _ in range(epochs):
        model.partial_fit(X, y, classes=[0, 1])
    return model


mistakes = [int((perceptron_after(k).predict(X) != y).sum())
            for k in range(1, EPOCHS + 1)]

figure, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))
for ax, epochs in [(axes[0], EARLY), (axes[1], LATE)]:
    model = perceptron_after(epochs)
    wrong = int((model.predict(X) != y).sum())
    show_regions(model, X, y, ax, f'After {epochs} epochs: {wrong} of 100 wrong')
    ax.legend(loc='upper left', fontsize=8)

axes[2].plot(range(1, EPOCHS + 1), mistakes, marker='s', ms=3, color='crimson')
axes[2].axvline(EARLY, color='gray', ls='--', lw=1)
axes[2].axvline(LATE, color='gray', ls='--', lw=1)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Flowers wrong, out of 100')
axes[2].set_title('It never settles')
axes[2].set_ylim(bottom=0)

plt.tight_layout()
plt.show()

print(f'epoch {EARLY}: {mistakes[EARLY - 1]:2} of 100 wrong')
print(f'epoch {LATE}: {mistakes[LATE - 1]:2} of 100 wrong')
print(f'range over {EPOCHS} epochs: {min(mistakes)} to {max(mistakes)}')

Notes:

## 2. Adaline: measure the error before the threshold

Adaline trains on the continuous output $z$ rather than on the thresholded
prediction, minimizing

$$L(\mathbf{w}, b) = \frac{1}{n}\sum_{i=1}^{n}\left(y^{(i)} - z^{(i)}\right)^2$$


In [ ]:
def adaline_losses(features, targets, eta0, epochs):
    """Train Adaline one epoch at a time, returning the loss after each."""
    model = SGDRegressor(loss='squared_error', penalty=None,
                         learning_rate='constant', eta0=eta0,
                         max_iter=1, tol=None, shuffle=False, random_state=0)
    history = []
    for _ in range(epochs):
        model.partial_fit(features, targets)
        history.append(float(((targets - model.predict(features)) ** 2).mean()))
    return model, history


adaline, loss_history = adaline_losses(X_std, y_pm, eta0=0.001, epochs=EPOCHS)

figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(range(1, EPOCHS + 1), mistakes, marker='s', ms=3, color='crimson')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Flowers wrong, out of 100')
axes[0].set_title('Perceptron: jagged, never settles')
axes[0].set_ylim(bottom=0)

axes[1].plot(range(1, EPOCHS + 1), loss_history, marker='o', ms=3,
             color='seagreen')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean squared error')
axes[1].set_title('Adaline: smooth, downhill, settles')

plt.tight_layout()
plt.show()

accuracy = (np.where(adaline.predict(X_std) >= 0, 1, -1) == y_pm).mean()
print(f'loss {loss_history[0]:.4f} -> {loss_history[-1]:.4f}, accuracy {accuracy:.3f}')

### with high learning rate

In [ ]:
_, exploding = adaline_losses(X, y_pm, eta0=0.08, epochs=12)   # raw, unscaled

print('loss per epoch, raw features, eta0 = 0.08')
for epoch, loss in enumerate(exploding, 1):
    print(f'  epoch {epoch:2}: {loss:.4g}')

## 3. Scaling



In [ ]:
def loss_surface(feature, targets, w_values, b_values):
    """Mean squared error over a grid of (w, b). Arithmetic, not a model."""
    grid_w, grid_b = np.meshgrid(w_values, b_values)
    predictions = feature[:, None, None] * grid_w + grid_b
    return grid_w, grid_b, ((targets[:, None, None] - predictions) ** 2).mean(axis=0)


figure, axes = plt.subplots(1, 2, figsize=(12.5, 5.6))
for ax, (title, feature) in zip(axes, [('Raw petal length', X[:, 0]),
                                       ('Standardized', X_std[:, 0])]):
    exact = LinearRegression().fit(feature.reshape(-1, 1), y_pm)
    centre_w, centre_b = float(exact.coef_[0]), float(exact.intercept_)
    half = 3.0
    grid_w, grid_b, surface = loss_surface(
        feature, y_pm,
        np.linspace(centre_w - half, centre_w + half, 200),
        np.linspace(centre_b - half, centre_b + half, 200))

    ax.contourf(grid_w, grid_b, surface, levels=35, cmap='viridis', alpha=0.9)
    ax.contour(grid_w, grid_b, surface, levels=18, colors='white', linewidths=0.4)
    ax.plot(centre_w, centre_b, '*', color='gold', ms=18, mec='black')
    ax.set_aspect('equal')
    ax.set_xlabel('weight $w$')
    ax.set_ylabel('bias $b$')

    design = np.c_[feature, np.ones(len(feature))]
    spread = np.linalg.eigvalsh(2.0 * design.T @ design / len(design))
    ax.set_title(f'{title}\ncanyon {np.sqrt(spread.max() / spread.min()):.0f} : 1')

plt.tight_layout()
plt.show()

In [ ]:
SHARED_ETA = 0.01

print(f'Adaline, eta0 = {SHARED_ETA}, identical in both rows\n')
print(f'{"":16}{"loss @ 10 epochs":>18}{"epochs to reach 0.30":>24}')
for title, features in [('Raw', X), ('Standardized', X_std)]:
    _, history = adaline_losses(features, y_pm, SHARED_ETA, 200)
    reached = next((i + 1 for i, v in enumerate(history) if v < 0.30), None)
    print(f'{title:16}{history[9]:>18.3f}{reached if reached else ">200":>24}')

print()
print(f'{"":16}{"stretch of the bowl":>21}')
for title, features in [('Raw', X), ('Standardized', X_std)]:
    design = np.c_[features, np.ones(len(features))]
    spread = np.linalg.eigvalsh(2.0 * design.T @ design / len(design))
    print(f'{title:16}{spread.max() / spread.min():>20,.0f}x')

## 4. Logistic regression

From here on $y \in \{-1, +1\}$, so the margin $m = y\,z$ is positive when the
prediction is right and larger when it is more confident.

Squash the output into a probability and use the loss that matches it:

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}, \qquad
L = -\frac{1}{n}\sum_{i=1}^{n}\left[y^{(i)} \log p^{(i)}
    + (1 - y^{(i)})\log(1 - p^{(i)})\right]$$

In [ ]:
shared = dict(penalty=None, max_iter=2000, tol=None, random_state=0)

adaline_sgd  = SGDClassifier(loss='squared_error', **shared).fit(X_std, y_pm)
logistic_sgd = SGDClassifier(loss='log_loss',      **shared).fit(X_std, y_pm)

for name, model in [('squared_error', adaline_sgd), ('log_loss', logistic_sgd)]:
    print(f"loss='{name}':{'':{14 - len(name)}}w = {model.coef_[0].round(3)}  "
          f"accuracy {model.score(X_std, y_pm):.3f}")

### Mean squared error punishes being too right

In [ ]:
rng = np.random.default_rng(0)
core_X = np.vstack([np.c_[rng.uniform(-6, 6, 45), rng.normal( 1.0, 0.35, 45)],
                    np.c_[rng.uniform(-6, 6, 45), rng.normal(-1.0, 0.35, 45)]])
core_y = np.r_[-np.ones(45), np.ones(45)]

far = np.c_[rng.normal(10, 0.5, 20), rng.normal(-14, 0.5, 20)]
both_X = np.vstack([core_X, far])
both_y = np.r_[core_y, np.ones(20)]

# LinearRegression gives the exact squared-error minimum, so the optimizer is
# never in question.
fits = {
    'least squares, without them': LinearRegression().fit(core_X, core_y),
    'least squares, with them':    LinearRegression().fit(both_X, both_y),
    'logistic, without them':      LogisticRegression(C=np.inf, max_iter=20000).fit(core_X, core_y),
    'logistic, with them':         LogisticRegression(C=np.inf, max_iter=20000).fit(both_X, both_y),
}


def line_of(model, span):
    w = np.ravel(model.coef_)
    b = float(np.ravel(model.intercept_)[0])
    return span, -(w[0] * span + b) / w[1]


def correct(model, features, labels):
    score = np.ravel(model.predict(features)) if isinstance(model, LinearRegression) \
        else model.decision_function(features)
    return (np.where(score >= 0, 1, -1) == labels).mean()


span = np.linspace(-8, 13, 2)
figure, ax = plt.subplots(figsize=(9, 6))
ax.scatter(*core_X[core_y < 0].T, c='tab:blue', s=26, label='class $-1$')
ax.scatter(*core_X[core_y > 0].T, c='tab:red', s=26, label='class $+1$')
ax.scatter(*far.T, c='tab:red', s=26, marker='^',
           label='class $+1$, far on the correct side')
for name, colour, style in [('least squares, without them', 'seagreen', '-'),
                            ('least squares, with them', 'crimson', '-'),
                            ('logistic, with them', 'black', '--')]:
    ax.plot(*line_of(fits[name], span), color=colour, ls=style, lw=2.4, label=name)
ax.set_ylim(-16, 6)
ax.legend(loc='lower left', fontsize=9)
ax.set_title('Adding points on the correct side breaks least squares')
plt.tight_layout()
plt.show()

print(f'{"":30}{"accuracy on the 90 core points":>31}')
for name, model in fits.items():
    print(f'{name:30}{correct(model, core_X, core_y):>31.3f}')


### Probabilities

In [ ]:
model = LogisticRegression(C=np.inf, max_iter=20000).fit(X_std, y_pm)

grid1, grid2 = np.meshgrid(np.linspace(-2.6, 2.6, 300),
                           np.linspace(-2.6, 2.6, 300))
probability = model.predict_proba(
    np.c_[grid1.ravel(), grid2.ravel()])[:, 1].reshape(grid1.shape)

figure, ax = plt.subplots(figsize=(7.5, 6))
shading = ax.contourf(grid1, grid2, probability, levels=20, cmap='RdBu_r',
                      alpha=0.8)
lines = ax.contour(grid1, grid2, probability, levels=[0.1, 0.5, 0.9],
                   colors='black', linewidths=[1, 2.5, 1],
                   linestyles=['dashed', 'solid', 'dashed'])
ax.clabel(lines, fmt='%.1f')
ax.scatter(*X_std[y_pm < 0].T, c='tab:blue', s=28, edgecolor='k', lw=0.4)
ax.scatter(*X_std[y_pm > 0].T, c='tab:red', s=28, edgecolor='k', lw=0.4)
figure.colorbar(shading, ax=ax, label='$P(\\mathrm{virginica} \\mid x)$')
ax.set_xlabel('Petal length [standardized]')
ax.set_ylabel('Petal width [standardized]')
ax.set_title('The 0.5 contour is the boundary. The others are the confidence.')
plt.tight_layout()
plt.show()

In [ ]:
setosa_X = df.iloc[:, [2, 3]].values.astype(float)
setosa_y = np.where(df.iloc[:, 4].values == 'Iris-setosa', 1, -1)

print('setosa vs the rest, which IS separable')
print(f'{"":34}{"||w||":>9}{"accuracy":>11}')
for tol in [1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12]:
    loose = LogisticRegression(C=np.inf, max_iter=200000, tol=tol)
    loose.fit(setosa_X, setosa_y)
    print(f'  unregularized, tol = {tol:>7.0e}  {np.linalg.norm(loose.coef_):9.2f}'
          f'{loose.score(setosa_X, setosa_y):11.3f}')

print()
for tol in [1e-4, 1e-8, 1e-12]:
    penalized = LogisticRegression(C=1.0, max_iter=200000, tol=tol)
    penalized.fit(setosa_X, setosa_y)
    print(f'  regularized,   tol = {tol:>7.0e}  {np.linalg.norm(penalized.coef_):9.2f}')

**Talking points.** Unregularized, the weights grow without limit: 5.5, then
13, 19, 27, 35, 41, and they would keep going. Accuracy is pinned at 1.000 the
whole way, so none of that growth buys a single correct prediction.

Add a penalty on $\lVert\mathbf{w}\rVert$ and the norm settles at 2.81 no
matter how long it runs, because now there is a genuine minimum. That is
regularization, and it is the next thing to cover.

## 5. The loss zoo

Every classifier in this demo is the same model, $\mathbf{w}^\top\mathbf{x} + b$.
They differ only in the loss they charge as a function of the margin $m = y\,z$.

With $y = \pm 1$ the Adaline loss really is a function of the margin alone:
$(y - z)^2 = (y(y - z))^2 = (1 - m)^2$, using $y^2 = 1$.


In [ ]:
ZOO = {
    'Perceptron':  ('perceptron',    lambda m: np.maximum(0.0, -m),      'crimson'),
    'Adaline/MSE': ('squared_error', lambda m: (1.0 - m) ** 2,           'seagreen'),
    'Logistic':    ('log_loss',      lambda m: np.logaddexp(0.0, -m),    'tab:blue'),
    'Linear SVM':  ('hinge',         lambda m: np.maximum(0.0, 1.0 - m), 'darkorange'),
}

margin = np.linspace(-3, 3, 400)
figure, ax = plt.subplots(figsize=(9, 5.5))
for name, (loss_name, curve, colour) in ZOO.items():
    ax.plot(margin, curve(margin), lw=2.5, color=colour, label=f"{name}  (loss='{loss_name}')")
ax.axvline(0, color='gray', lw=0.8, ls=':')
ax.set_ylim(-0.2, 6)
ax.set_xlabel('margin $m = y\\,z$      (negative = wrong, positive = right)')
ax.set_ylabel('loss charged')
ax.set_title('Four classifiers, one model, four losses')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (name, (loss_name, _, _)) in zip(axes.ravel(), ZOO.items()):
    model = SGDClassifier(loss=loss_name, penalty=None, max_iter=2000,
                          tol=None, random_state=0).fit(X_std, y_pm)
    show_regions(model, X_std, y_pm, ax,
                 f"{name}\nloss='{loss_name}', accuracy {model.score(X_std, y_pm):.3f}")
plt.tight_layout()
plt.show()

| Classifier | Loss $\ell(m)$ | Behavior |
| --- | --- | --- |
| Perceptron | $\max(0, -m)$ | Zero the instant it is correct, so it does not care how correct |
| Adaline | $(1 - m)^2$ | Penalizes being wrong **and** being too right |
| Logistic | $\log(1 + e^{-m})$ | Smooth, decays toward zero, never quite reaches it |
| Linear SVM | $\max(0, 1 - m)$ | Zero only once past a margin of 1 |



> A new training point lands **far on the correct side** of the boundary.
> Which of the four classifiers move their boundary, and which do not?


**Reflection questions**

1. Why does Adaline converge on data that is not separable when the perceptron
   does not?
2. Why does scaling speed up gradient descent?
3. Why is the mean squared error a poor loss for classification?